In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import uuid
import time

options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

product_urls = {
    "Toner mist": "https://kaspi.kz/shop/p/skin1004-mist-hyalu-cica-cloudy-120-ml-104904631/?0&c=750000000&ref=shared_link",
    "Dr.Althea 345": "https://kaspi.kz/shop/p/dr-althea-krem-345-relief-dlja-litsa-50-ml-115616909/?0&c=750000000&ref=shared_link",
    "Lagom": "https://kaspi.kz/shop/p/lagom-krem-cellus-mild-moisture-dlja-litsa-80-ml-100676372/?0&c=750000000&ref=shared_link",
    "telescopic тушь": "https://kaspi.kz/shop/p/tush-dlja-resnits-loreal-paris-telescopic-explosion-dlja-ob-ema-udlinjajuschaja-chernyi-17400312/?c=750000000&ref=shared_link",
    "Бальзам": "https://kaspi.kz/shop/p/fito-kosmetik-bal-zam-vazelin-uvlazhnenie-i-zaschita-10-102666500/?c=750000000&sr=6&qid=d4e7e5efef9988a52ecc9c155959dd20&ref=shared_link",
    "relouis пудра ": "https://kaspi.kz/shop/p/pudra-relouis-hd-powder-fixing-transparent-rassypchataja-01-belyi-100968225/?c=750000000&sr=1&qid=d285461498d0d5703498499264187614&ref=shared_link"
}

all_reviews = []

for product_name, url in product_urls.items():
    driver.get(url)
    time.sleep(4)
    collected = 0

    while True:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        reviews = soup.find_all("div", class_="reviews__review")

        for review in reviews:
            if collected >= 100:  # Көбірек пікір жинау үшін 100-ге дейін көбейтуге болады
                break

            review_id = str(uuid.uuid4())

            date_tag = review.find("div", class_="reviews__date")
            date = date_tag.text.strip() if date_tag else None

            user_tag = review.find("div", class_="reviews__author")
            user = user_tag.text.strip() if user_tag else "Аноним"

            comment_tag = review.find("div", class_="reviews__review-text")
            comment = comment_tag.get_text(strip=True).replace("Комментарий:", "") if comment_tag else None

            rating_tag = review.find("div", class_=lambda c: c and "rating" in c)
            rating = None
            if rating_tag:
                classes = rating_tag.get("class")
                for cls in classes:
                    if cls.startswith("_"):
                        try:
                            rating = int(cls[1:]) / 10
                        except ValueError:
                            rating = None

            all_reviews.append({
                "ID": review_id,
                "Атауы": product_name,
                "Күні": date,
                "Пайдаланушы": user,
                "Баға": rating,
                "Пікір": comment
            })

            collected += 1

        # 👉 Келесі бетке өту батырмасын іздеу
        try:
            next_button = driver.find_element(By.XPATH, "//button[contains(@class, 'reviews__load-more')]")
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(3)
        except:
            break  # Егер "келесі" батырмасы жоқ болса, циклді тоқтату

driver.quit()

df = pd.DataFrame(all_reviews)
df.to_csv("Kaspi_pikirleri.csv", index=False, encoding='utf-8-sig')
print(f"✅ {len(all_reviews)} пікір сәтті сақталды!")


✅ 54 пікір сәтті сақталды!


In [2]:
df = pd.read_csv("Kaspi_pikirleri.csv")
df

,ID,Атауы,Күні,Пайдаланушы,Баға,Пікір
0,488fb3de-f7c2-4d1b-b14d-99543968c330,Toner mist,17.06.2024,Ирина,5.0,Впервые пробую этот мист. Мне понравился. Осве...
1,1b3b60e9-51dc-4d76-9bdb-3d88ce5e86c4,Toner mist,28.02.2023,Наргиза,4.0,"Достоинства:Удобно брызгать, расход небольшой...."
2,e2c66e91-c2fb-4c0b-8a0e-0c9753e8715d,Toner mist,23.09.2024,Нұрайлым,5.0,"Құрбымнан көріп алғанмын, бетім құрғақ болғасы..."
3,37a65da0-7f3d-429b-a2ae-def93e40fa14,Toner mist,13.07.2024,Ақерке,5.0,"Очень классная, качество супер, советую взять...."
4,238229b0-071c-4398-bc3c-ce548c0a75bc,Toner mist,24.03.2024,Гульзада,3.0,"Мист шашатын жері істен шыққан, брак, сатушыға..."
5,ac09a7fc-7f9f-4871-a5f8-0b7d2bc9b943,Toner mist,08.06.2023,Айгерім,5.0,Достоинства:Реально облачный спрей1 человек(а)...
6,0fdd32ef-b474-4593-b484-a996f657b63c,Toner mist,14.04.2025,Рита,5.0,"Немного шиплет от того, что сразу даёт эффекты..."
7,c45a1cf0-4af2-4e59-b8c2-4eed3c90fe55,Toner mist,04.02.2025,Коркем,5.0,Вся серия Skin 1004 супер! Каспи как всегда лу...
8,f317b301-b2f6-4c1c-93ae-ad87aba3e66f,Toner mist,17.01.2025,Сымбат,5.0,Товар оригинал! Спасибо большое магазину Korea...
9,e15fab76-5f82-44e9-943e-ef56481eec3e,Dr.Althea 345,15.02.2024,Лилия,5.0,"Самый лучший, лёгкий, не вызывает аллергию, пр..."


In [5]:
import re
from langdetect import detect

# Функция пікірдің тілін тексеру және қазақ/орыс тілін анықтау
def is_valid_language(text):
    try:
        language = detect(text)
        return language in ['kk', 'ru']  # Тек қазақ және орыс тілдерін қалдырамыз
    except:
        return False

# Пікірді тазарту: артық сөздер мен символдарды алып тастау
def clean_text(text):
    if not text:
        return ""
    # Артық символдарды алып тастау
    text = re.sub(r'\s+', ' ', text)  # Көп бос орынды бір бос орынмен ауыстыру
    text = re.sub(r'[^\w\s,.-]', '', text)  # Тек әріптер мен сандар, үтір, нүкте, сызықшалар қалдыру
    return text.strip()

# Тазарту процесі
df['Пікір'] = df['Пікір'].apply(lambda x: clean_text(x) if is_valid_language(x) else "")


In [6]:
# Пікір ұзындығын есептеу
df['Пікір Ұзындығы'] = df['Пікір'].apply(lambda x: len(str(x)) if x else 0)

# Баға бағанын сандық форматқа түрлендіру
df['Баға'] = pd.to_numeric(df['Баға'], errors='coerce')  # Қате мәндерді NaN деп қалдырады


In [7]:
# Деректерді CSV файлында сақтау
df.to_csv("Kaspi_pikirleri_cleaned.csv", index=False, encoding='utf-8-sig')

In [8]:
df = pd.read_csv("Kaspi_pikirleri_cleaned.csv")
df

,ID,Атауы,Күні,Пайдаланушы,Баға,Пікір,Пікір Ұзындығы
0,488fb3de-f7c2-4d1b-b14d-99543968c330,Toner mist,17.06.2024,Ирина,5.0,Впервые пробую этот мист. Мне понравился. Осве...,238
1,1b3b60e9-51dc-4d76-9bdb-3d88ce5e86c4,Toner mist,28.02.2023,Наргиза,4.0,"ДостоинстваУдобно брызгать, расход небольшой.Н...",132
2,e2c66e91-c2fb-4c0b-8a0e-0c9753e8715d,Toner mist,23.09.2024,Нұрайлым,5.0,"Құрбымнан көріп алғанмын, бетім құрғақ болғасы...",194
3,37a65da0-7f3d-429b-a2ae-def93e40fa14,Toner mist,13.07.2024,Ақерке,5.0,"Очень классная, качество супер, советую взять....",81
4,238229b0-071c-4398-bc3c-ce548c0a75bc,Toner mist,24.03.2024,Гульзада,3.0,"Мист шашатын жері істен шыққан, брак, сатушыға...",225
5,ac09a7fc-7f9f-4871-a5f8-0b7d2bc9b943,Toner mist,08.06.2023,Айгерім,5.0,ДостоинстваРеально облачный спрей1 человека по...,68
6,0fdd32ef-b474-4593-b484-a996f657b63c,Toner mist,14.04.2025,Рита,5.0,"Немного шиплет от того, что сразу даёт эффекты...",97
7,c45a1cf0-4af2-4e59-b8c2-4eed3c90fe55,Toner mist,04.02.2025,Коркем,5.0,Вся серия Skin 1004 супер Каспи как всегда луч...,85
8,f317b301-b2f6-4c1c-93ae-ad87aba3e66f,Toner mist,17.01.2025,Сымбат,5.0,Товар оригинал Спасибо большое магазину Korean...,171
9,e15fab76-5f82-44e9-943e-ef56481eec3e,Dr.Althea 345,15.02.2024,Лилия,5.0,"Самый лучший, лёгкий, не вызывает аллергию, пр...",232
